In [11]:
# importo todo para el pipeline completo
import cv2
import numpy as np
import pickle
import time
import datetime
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from ultralytics import YOLO
from insightface.app import FaceAnalysis
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# --- configuración global ---
CAMARA_INDEX  = 0
UMBRAL_COSENO = 0.5       # por debajo = intruso
CAMARA_INDEX  = 0
MODELO_YOLO   = "models/yolov12n-face.pt"
WHITELIST_PATH = Path("whitelist/whitelist.pkl")
ALERTS_PATH    = Path("alerts_log")

# --- cargo YOLO ---
yolo = YOLO(MODELO_YOLO)
print("✅ YOLO cargado")

# --- cargo InsightFace ---
insight = FaceAnalysis(name="buffalo_l", providers=["CPUExecutionProvider"])
insight.prepare(ctx_id=-1, det_size=(640, 640))
print("✅ InsightFace cargado")

# --- cargo whitelist ---
with open(WHITELIST_PATH, "rb") as f:
    whitelist = pickle.load(f)

print(f"✅ Whitelist cargada: {list(whitelist.keys())}")

✅ YOLO cargado
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/mynorhm/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/mynorhm/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/mynorhm/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/mynorhm/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/mynorhm/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3,

In [9]:
import urllib.request
import json

# --- configuración de ntfy ---
# 1. descarga la app ntfy en tu móvil (iOS o Android)
# 2. pulsa + y suscríbete a un canal con el nombre que quieras (este ej usa: SistemaSeguridad2026)
# 3. ese mismo nombre va aquí abajo
# 4. sin registro, sin cuenta, gratis hasta 250 mensajes/día
NTFY_CANAL = "SistemaSeguridad2026"

def enviar_alerta_ntfy(nombre, score):
    try:
        ts = datetime.datetime.now().strftime("%d/%m/%Y %H:%M:%S")

        req = urllib.request.Request(
            f"https://ntfy.sh/{NTFY_CANAL}",
            data=f"Cara no reconocida a las {ts} — Score: {score:.3f}".encode("utf-8"),
            headers={
                "Title"   : "Intruso detectado",
                "Priority": "high",
                "Tags"    : "rotating_light,camera"
            },
            method="POST"
        )
        urllib.request.urlopen(req)
        print(f"📱 Notificación enviada a {NTFY_CANAL}")

    except Exception as e:
        print(f"❌ Error enviando notificación: {e}")

print("✅ Sistema de alertas ntfy listo")

✅ Sistema de alertas ntfy listo


In [15]:
# todas las funciones que necesita el pipeline en un solo lugar

def extraer_embeddings(frame, caras_bbox):
    """dado un frame y los bboxes de YOLO, extraigo embeddings con InsightFace"""
    faces = insight.get(frame)
    return faces  # InsightFace detecta sola, usamos YOLO solo para velocidad

def reconocer(faces):
    """comparo cada cara contra la whitelist y devuelvo resultados"""
    resultados = []
    for face in faces:
        emb = face.embedding.reshape(1, -1)
        mejor_nombre = "Intruso"
        mejor_score  = 0.0

        for nombre, emb_ref in whitelist.items():
            score = cosine_similarity(emb, emb_ref.reshape(1, -1))[0][0]
            if score > mejor_score:
                mejor_score  = score
                mejor_nombre = nombre if score >= UMBRAL_COSENO else "Intruso"

        resultados.append({
            "nombre"    : mejor_nombre,
            "score"     : mejor_score,
            "bbox"      : face.bbox.astype(int),
            "autorizado": mejor_nombre != "Intruso"
        })
    return resultados

def dibujar(frame, resultados):
    """dibujo bboxes y etiquetas sobre el frame"""
    out = frame.copy()
    for r in resultados:
        x1, y1, x2, y2 = r["bbox"]
        color = (0, 255, 0) if r["autorizado"] else (0, 0, 255)
        label = f"{r['nombre']} ({r['score']:.2f})"

        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(out, (x1, y1 - h - 10), (x1 + w, y1), color, -1)
        cv2.putText(out, label, (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    return out

def guardar_alerta(frame, nombre, score, resultados):
    """guardo el frame del intruso con timestamp en alerts_log y envia mail de alerta"""
    ts    = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = ALERTS_PATH / f"alerta_{ts}_{nombre}.jpg"

    # guardo el frame con los bboxes y labels dibujados
    frame_viz = dibujar(frame, resultados)
    cv2.imwrite(str(fname), frame_viz)

    print(f"🚨 ALERTA guardada: {fname}")
    enviar_alerta_ntfy(nombre, score)  # Notificacion push de alerta de intruso
    return fname

print("✅ Funciones listas")

✅ Funciones listas


In [16]:
# loop principal del sistema de seguridad
# presiona Q para salir

print("🎥 Iniciando sistema de seguridad... (Q para salir)")

cap = cv2.VideoCapture(CAMARA_INDEX)
ultimo_alerta = {}  # evito spam de alertas por persona
COOLDOWN_SEGUNDOS = 10  # mínimo tiempo entre alertas del mismo intruso

while True:
    ret, frame = cap.read()
    if not ret:
        print("❌ Error leyendo frame")
        break

    # detecto y reconozco
    faces = insight.get(frame)
    resultados = reconocer(faces)

    # proceso alertas
    ahora = time.time()
    for r in resultados:
        if not r["autorizado"]:
            nombre = r["nombre"]
            ultimo = ultimo_alerta.get(nombre, 0)
            if ahora - ultimo > COOLDOWN_SEGUNDOS:
                guardar_alerta(frame, nombre, r["score"], resultados)
                ultimo_alerta[nombre] = ahora

    # dibujo y muestro
    frame_viz = dibujar(frame, resultados)
    cv2.imshow("Sistema de Seguridad", frame_viz)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        print("👋 Cerrando sistema...")
        break

cap.release()
cv2.destroyAllWindows()
print("✅ Sistema cerrado")

🎥 Iniciando sistema de seguridad... (Q para salir)
🚨 ALERTA guardada: alerts_log/alerta_20260611_204941_Intruso.jpg
📱 Notificación enviada a SistemaSeguridad2026
👋 Cerrando sistema...
✅ Sistema cerrado


In [2]:
! uv add fastai toml

Resolved 226 packages in 346ms                                       
Prepared 1 package in 34ms                                               
Installed 1 package in 1ms                                  
 + toml==0.10.2


In [5]:
from huggingface_hub import hf_hub_download

ruta = hf_hub_download(
    repo_id="mynorhm/security-room-classifier",
    filename="full_model.pth",
    repo_type="model"
)

import shutil
shutil.copy(ruta, "models/full_model.pth")
print("✅ full_model.pth descargado en models/")

full_model.pth:   0%|          | 0.00/201M [00:00<?, ?B/s]

✅ full_model.pth descargado en models/


In [6]:
ruta_clases = hf_hub_download(
    repo_id="mynorhm/security-room-classifier",
    filename="clases.json",
    repo_type="model"
)

shutil.copy(ruta_clases, "models/clases.json")
print("✅ clases.json descargado en models/")

clases.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

✅ clases.json descargado en models/


In [8]:
!uv add timm

Resolved 228 packages in 331ms                                       
Prepared 2 packages in 197ms                                             
Installed 2 packages in 4ms                                 
 + safetensors==0.8.0
 + timm==1.0.27


In [9]:
import torch
import json
from fastai.vision.all import *

# cargo el modelo y las clases
learn_zonas = torch.load("models/full_model.pth", map_location="cpu", weights_only=False)
learn_zonas.eval()

with open("models/clases.json") as f:
    clases_zonas = json.load(f)

print(f"✅ Modelo cargado")
print(f"✅ Clases: {clases_zonas}")

✅ Modelo cargado
✅ Clases: ['bathroom', 'bedroom', 'corridor', 'dining_room', 'garage', 'kitchen', 'livingroom', 'stairscase']
